# 05 — Social Media Posting (Phase 1b)

Validate direct posting to Mastodon and Bluesky APIs.

**Prerequisites:**
- Mastodon: Create app at mastodon.social → Settings → Development
  - Scopes: `read:accounts`, `read:statuses`, `write:statuses`
  - Add `MASTODON_ACCESS_TOKEN` to `.env`
- Bluesky: Generate app password at bsky.app → Settings → App Passwords
  - Add `BLUESKY_APP_PASSWORD` to `.env`

> **Do not run this notebook until auth tokens are configured.**

In [1]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
import os

load_dotenv('../.env')

MASTODON_ACCESS_TOKEN = os.getenv('MASTODON_ACCESS_TOKEN')
BLUESKY_APP_PASSWORD = os.getenv('BLUESKY_APP_PASSWORD')
BLUESKY_HANDLE = os.getenv('BLUESKY_HANDLE', 'fretchen.eu')
MASTODON_INSTANCE = os.getenv('MASTODON_INSTANCE', 'https://mastodon.social')

print(f'Mastodon token: {"✅" if MASTODON_ACCESS_TOKEN else "❌ NOT SET"}')
print(f'Bluesky password: {"✅" if BLUESKY_APP_PASSWORD else "❌ NOT SET"}')
print(f'Mastodon instance: {MASTODON_INSTANCE}')
print(f'Bluesky handle: {BLUESKY_HANDLE}')

Mastodon token: ✅
Bluesky password: ✅
Mastodon instance: https://mastodon.social
Bluesky handle: fretchen.eu


## 1. Mastodon — Verify Credentials via MastodonClient

In [2]:
from agent.platforms.mastodon import MastodonClient

if not MASTODON_ACCESS_TOKEN:
    print('⏭️ Skipping Mastodon — no token configured.')
    masto = None
else:
    masto = MastodonClient(instance=MASTODON_INSTANCE, access_token=MASTODON_ACCESS_TOKEN)
    account = masto.verify_credentials()
    print(f'Logged in as: @{account["acct"]}')
    print(f'Followers: {account["followers_count"]}')
    print(f'Following: {account["following_count"]}')
    print(f'Posts: {account["statuses_count"]}')

Logged in as: @fretchen
Followers: 3
Following: 21
Posts: 19


## 2. Mastodon — Post a Test Status

⚠️ This will actually post to Mastodon. Use `visibility: 'direct'` for testing
(only visible to you).

In [3]:
if not masto:
    print('⏭️ Skipping.')
else:
    # Post with 'direct' visibility — only you can see it
    test_post = masto.post_status(
        'Growth Agent test post — please ignore. 🔬',
        visibility='direct',
    )
    print(f'Posted! ID: {test_post["id"]}')
    print(f'URL: {test_post["url"]}')

    # Clean up
    masto.delete_status(test_post['id'])
    print('Deleted test post.')

Posted! ID: 116397769244645273
URL: https://mastodon.social/@fretchen/116397769244645273
Deleted test post.


## 3. Bluesky — Authenticate via BlueskyClient

In [4]:
from agent.platforms.bluesky import BlueskyClient

if not BLUESKY_APP_PASSWORD:
    print('⏭️ Skipping Bluesky — no app password configured.')
    bsky = None
else:
    bsky = BlueskyClient(handle=BLUESKY_HANDLE, app_password=BLUESKY_APP_PASSWORD)
    profile = bsky.get_profile()
    print(f'Authenticated as: {profile["handle"]}')
    print(f'Followers: {profile.get("followersCount", 0)}')
    print(f'Following: {profile.get("followsCount", 0)}')
    print(f'Posts: {profile.get("postsCount", 0)}')

Authenticated as: fretchen.eu
Followers: 8
Following: 33
Posts: 44


## 4. Bluesky — Post a Test Status

⚠️ Bluesky has no 'direct' visibility. This creates a real post.
We immediately delete it after.

In [5]:
if not bsky:
    print('⏭️ Skipping.')
else:
    # Bluesky has no 'direct' visibility — post and immediately delete
    result = bsky.post('Growth Agent test post — please ignore.')
    rkey = result['uri'].split('/')[-1]
    print(f'Posted! URI: {result["uri"]}')

    bsky.delete_post(rkey)
    print('Deleted test post.')

Posted! URI: at://did:plc:t3362qxvovwfdx6fhw4ux642/app.bsky.feed.post/3mjeztco7u525
Deleted test post.


## 5. Publish a Draft from Content Queue

Load a pending draft and publish it to the correct platform.
Uses the `publish_draft()` helper from `agent/publisher.py`.

In [6]:
from agent.storage import LocalStorage
from agent.models import ContentQueue

store = LocalStorage(base_dir='../state')
queue_data = store.read('content_queue.json')

if queue_data:
    queue = ContentQueue(**queue_data)
    print(f'Pending drafts: {len(queue.drafts)}')
    print(f'Approved drafts: {len(queue.approved)}')
    for d in queue.drafts:
        print(f'  - {d.id} | {d.channel} ({d.language}) | {len(d.content)} chars')
        print(f'    {d.content[:100]}...')
else:
    print('No content queue — run notebook 03 first.')

Pending drafts: 2
Approved drafts: 1
  - draft_9a82a5a4 | bluesky (en) | 173 chars
    "Unlocking growth on Optimism and Base: get the inside scoop from an independent x402 facilitator ht...
  - draft_8e4b825f | mastodon (de) | 311 chars
    Kann ein unabhängiger Facilitator wirklich den Erfolg von Optimism und Base entscheiden? Ein interes...


In [7]:
from agent.publisher import publish_draft

# Publish all approved drafts that are ready (scheduled_at <= now)
clients = {}
if masto:
    clients['mastodon'] = masto
if bsky:
    clients['bluesky'] = bsky

if not queue_data:
    print('No queue loaded.')
elif not queue.approved:
    print('No approved drafts. Use notebook 06 to review and approve drafts first.')
else:
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    ready = [d for d in queue.approved if d.scheduled_at and d.scheduled_at <= now]
    not_ready = [d for d in queue.approved if not d.scheduled_at or d.scheduled_at > now]

    print(f'{len(ready)} draft(s) ready to publish, {len(not_ready)} scheduled for later.')

    for draft in ready:
        if draft.channel not in clients:
            print(f'⏭️ Skipping {draft.id} — no {draft.channel} client.')
            continue
        result = publish_draft(draft, clients[draft.channel])
        print(f'✅ Published {draft.id} to {draft.channel}: {result}')
        # Move from approved to published
        queue.approved.remove(draft)
        draft.status = 'published'
        queue.published.append(draft)

    store.write('content_queue.json', queue)
    print(f'Queue updated. Published: {len(queue.published)}, Approved: {len(queue.approved)}')

0 draft(s) ready to publish, 1 scheduled for later.
Queue updated. Published: 0, Approved: 1
